In [81]:
import numpy as np
import csv
import matplotlib.pyplot as plt
import pandas as pd
import pathlib as Path

In [82]:
DAYS_PER_MONTH = 30
HOURS_PER_DAY = 16
N_LANES = 50
E_B = 1.0

In [83]:
# Searches your user directory (Desktop, Documents, Downloads, repos, etc.) for airport.csv
matches = list(Path.Path.home().rglob("airport.csv"))
print("Found files:", matches)

Found files: [PosixPath('/Users/cesare/Downloads/airport.csv')]


In [84]:
df = pd.read_csv(matches[0])

In [85]:
with open("/Users/cesare/Downloads/airport.csv", newline="", encoding="utf-8") as csvfile:
    rows = list(csv.DictReader(csvfile))

september_rows = [
    row for row in rows
    if row["Month"] == "September" and int(row["Year"]) <= 2019
]

numeric_columns = [
    column for column in rows[0]
    if column not in ("Year", "Month")
]

september_average = {
    column: np.mean([int(row[column]) for row in september_rows])
    for column in numeric_columns
}

print(f"September rows: {len(september_rows)}")
for column, average in september_average.items():
    print(f"{column}: {average:.2f}")

September rows: 6
Europe Flights: 34973.67
Intercontinental Flights: 7698.17
Total Flights: 42671.83
Europe Passengers: 4246296.67
Intercontinental Passengers: 1698327.17
Total Passengers: 5944623.83


In [86]:
df

,Year,Month,Europe Flights,Intercontinental Flights,Total Flights,Europe Passengers,Intercontinental Passengers,Total Passengers
0,2014,January,24728,6664,31392,2301528,1315024,3616552
1,2014,February,23394,6056,29450,2289187,1146405,3435592
2,2014,March,27140,6893,34033,2744289,1332183,4076472
3,2014,April,29742,6888,36630,3168885,1391757,4560642
4,2014,May,32356,7137,39493,3537385,1450566,4987951
...,...,...,...,...,...,...,...,...
127,2024,August,35277,7841,43118,4618422,1798617,6417039
128,2024,September,33180,8598,41778,4170524,1860289,6030813
129,2024,October,34487,7518,42005,4371263,1667703,6038966
130,2024,November,29574,7014,36588,3572442,1498744,5071186


In [87]:
pre_covid = df.iloc[0:][0:72]

In [88]:
pre_covid

,Year,Month,Europe Flights,Intercontinental Flights,Total Flights,Europe Passengers,Intercontinental Passengers,Total Passengers
0,2014,January,24728,6664,31392,2301528,1315024,3616552
1,2014,February,23394,6056,29450,2289187,1146405,3435592
2,2014,March,27140,6893,34033,2744289,1332183,4076472
3,2014,April,29742,6888,36630,3168885,1391757,4560642
4,2014,May,32356,7137,39493,3537385,1450566,4987951
...,...,...,...,...,...,...,...,...
67,2019,August,36867,8255,45122,4804762,2005997,6810759
68,2019,September,36193,7938,44131,4649227,1844241,6493468
69,2019,October,36508,7995,44503,4600170,1831273,6431443
70,2019,November,30599,7203,37802,3713816,1609774,5323590


In [89]:
means = pre_covid.mean(numeric_only=True)

In [90]:
for i in means:
    for name, value in means.items():
        print(f"{name}: {value:.0f}")

Year: 2016
Europe Flights: 32281
Intercontinental Flights: 7453
Total Flights: 39734
Europe Passengers: 3790002
Intercontinental Passengers: 1595524
Total Passengers: 5385526
Year: 2016
Europe Flights: 32281
Intercontinental Flights: 7453
Total Flights: 39734
Europe Passengers: 3790002
Intercontinental Passengers: 1595524
Total Passengers: 5385526
Year: 2016
Europe Flights: 32281
Intercontinental Flights: 7453
Total Flights: 39734
Europe Passengers: 3790002
Intercontinental Passengers: 1595524
Total Passengers: 5385526
Year: 2016
Europe Flights: 32281
Intercontinental Flights: 7453
Total Flights: 39734
Europe Passengers: 3790002
Intercontinental Passengers: 1595524
Total Passengers: 5385526
Year: 2016
Europe Flights: 32281
Intercontinental Flights: 7453
Total Flights: 39734
Europe Passengers: 3790002
Intercontinental Passengers: 1595524
Total Passengers: 5385526
Year: 2016
Europe Flights: 32281
Intercontinental Flights: 7453
Total Flights: 39734
Europe Passengers: 3790002
Intercontinen

In [91]:
df = pd.read_csv(matches[0])
sep = df[(df["Month"] == "September") & (df["Year"].between(2015, 2019))]

In [93]:
P_bar = sep["Total Passengers"].mean()                 
per_hour_airport = P_bar / (DAYS_PER_MONTH * HOURS_PER_DAY)
per_hour_lane = per_hour_airport / N_LANES
lam = per_hour_lane / 60                              

rho_baseline = lam * E_B # 1 server
rho_option_a = lam * E_B / 2 # 2 servers

In [ ]:
print(sep[["Year", "Total Passengers"]].to_string(index=False))
print(f"\nAverage September passengers : {P_bar:,.1f}")
print(f"Airport-wide arrivals / hour : {per_hour_airport:,.2f}")
print(f"Per-lane arrivals / hour     : {per_hour_lane:,.2f}")
print(f"lambda (per lane, per min)   : {lam:.4f}")
print(f"rho baseline / Option B (c=1): {rho_baseline:.4f}")
print(f"rho Option A (c=2)           : {rho_option_a:.4f}")

 Year  Total Passengers
 2015           5466239
 2016           5866487
 2017           6259104
 2018           6396940
 2019           6493468

Average September passengers : 6,096,447.6
Airport-wide arrivals / hour : 12,700.93
Per-lane arrivals / hour     : 254.02
lambda (per lane, per min)   : 4.2336
rho baseline / Option B (c=1): 4.2336
rho Option A (c=2)           : 2.1168


In [94]:
import heapq
import numpy as np
from scipy import stats

In [99]:
ARRIVAL, DEPARTURE = 0, 1

In [100]:
def sample_service(rng, mean, sd):
    s = rng.normal(mean, sd)
    while s < 0:
        s = rng.normal(mean, sd)
    return s

In [101]:
def simulate(lam, mean_s, sd_s, n_servers, n_passengers, seed=None):
    rng = np.random.default_rng(seed)
    fel = []                                  
    waits = np.empty(n_passengers)
    arrival_time = np.empty(n_passengers)
    queue = []                                
    q_head = 0
    busy = 0
    n_arrived = 0

    # schedule first arrival
    heapq.heappush(fel, (rng.exponential(1.0 / lam), ARRIVAL, 0))

    while fel:
        t, etype, pid = heapq.heappop(fel)

        if etype == ARRIVAL:
            arrival_time[pid] = t
            n_arrived += 1
            if n_arrived < n_passengers:      
                heapq.heappush(fel, (t + rng.exponential(1.0 / lam), ARRIVAL, pid + 1))
            if busy < n_servers:             
                busy += 1
                waits[pid] = 0.0
                heapq.heappush(fel, (t + sample_service(rng, mean_s, sd_s), DEPARTURE, pid))
            else:
                queue.append(pid)

        else:  # departure
            if q_head < len(queue):         
                nxt = queue[q_head]
                q_head += 1
                waits[nxt] = t - arrival_time[nxt]
                heapq.heappush(fel, (t + sample_service(rng, mean_s, sd_s), DEPARTURE, nxt))
            else:
                busy -= 1

    return waits

In [102]:
E_B, SIGMA = 1.0, 0.25
RHO_TARGET = 0.85
R = 40                  
WARMUP = 1000          
N_PER_REP = 51_000      
ALPHA = 0.05

lam_test = RHO_TARGET / E_B
rho_test = lam_test * E_B

assert rho_test < 1

ES2 = E_B**2 + SIGMA**2
Wq_theory = lam_test * ES2 / (2 * (1 - rho_test))
seeds = np.random.SeedSequence(2025).spawn(R)

rep_means = np.array([
    simulate(lam_test, E_B, SIGMA, 1, N_PER_REP, seed=s)[WARMUP:].mean()
    for s in seeds
])
Wq_sim = rep_means.mean()
sd = rep_means.std(ddof=1)
se = sd / np.sqrt(R)

t_stat, p_val = stats.ttest_1samp(rep_means, Wq_theory)
t_crit = stats.t.ppf(1 - ALPHA / 2, R - 1)
ci = (Wq_sim - t_crit * se, Wq_sim + t_crit * se)

print(f"lambda_test = {lam_test:.3f}, rho_test = {rho_test:.3f}")
print(f"E[S^2] = {ES2:.4f}")
print(f"Wq_theory (P-K)      = {Wq_theory:.4f} min")
print(f"Wq_sim (mean of {R})  = {Wq_sim:.4f} min   (sd of rep means = {sd:.4f}, SE = {se:.4f})")
print(f"95% CI for Wq_sim    = ({ci[0]:.4f}, {ci[1]:.4f})")
print(f"t = {t_stat:.4f}, df = {R-1}, critical |t| = {t_crit:.4f}, p-value = {p_val:.4f}")
print("Reject H0" if p_val < ALPHA else "Fail to reject H0 -> simulator validated at alpha=0.05")


lambda_test = 0.850, rho_test = 0.850
E[S^2] = 1.0625
Wq_theory (P-K)      = 3.0104 min
Wq_sim (mean of 40)  = 3.0135 min   (sd of rep means = 0.1746, SE = 0.0276)
95% CI for Wq_sim    = (2.9576, 3.0693)
t = 0.1114, df = 39, critical |t| = 2.0227, p-value = 0.9119
Fail to reject H0 -> simulator validated at alpha=0.05
